In [1]:
import os
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

d:\Program\envs\agent_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
C:\Users\ANH TU\AppData\Local\Temp\ipykernel_36532\240993011.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

vectorstore = FAISS.load_local(
    r"D:\PharmaRAG-VN\Data\VectorStore\faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [3]:
# Query
query = "Viết giúp tôi lá thư gửi thầy xin nghỉ học"

# Retrieve Top 5
docs = vectorstore.similarity_search(
    query,
    k=5
)

context = "\n\n".join(
    f"Metadata: {doc.page_content}"
    for doc in docs
)

print(context)

Metadata: Document Type: monograph
Level 1: **BISACODYL**
Level 2: **Thai kỳ và cho con bú**
Content:
### **Thời kỳ mang thai**  
Hiện nay chưa có dữ liệu đáng tin cậy về thuốc gây quái thai ở súc vật. Trong lâm sàng, hiện nay chưa có dữ liệu thích đáng đầy đủ để đánh giá bisacodyl gây dị dạng hoặc độc cho thai khi dùng bisacodyl cho người mang thai. Sử dụng an toàn bisacodyl tannex trong khi mang thai cũng chưa được xác định. Do đó, không nên dùng bisacodyl cho phụ nữ mang thai. Nếu dùng, phải theo dõi cẩn thận.  
### **Thời kỳ cho con bú**  
Thuốc qua sữa với một lượng rất nhỏ. Rất thận trọng dùng thuốc cho người mẹ đang cho con bú.

Metadata: Document Type: monograph
Level 1: **AZITHROMYCIN**
Level 2: **Liều lượng và cách dùng**
Content:
### *Cách dùng:*  
Azithromycin có thể uống hoặc tiêm truyền tĩnh mạch, không được tiêm thẳng vào tĩnh mạch hoặc tiêm bắp.  
Thuốc uống: Viên thông thường hoặc hỗn dịch uống thông thường chứa 100 mg hoặc 200 mg azithromycin trong 5 ml hoặc chứa một 

In [4]:
prompt_template = """
Bạn là một dược sĩ lâm sàng giàu kinh nghiệm. Hãy sử dụng CÁC THÔNG TIN TRONG PHẦN NGỮ CẢNH dưới đây để trả lời câu hỏi của người dùng một cách chính xác và dễ hiểu.
Nếu thông tin trong Ngữ cảnh không đủ để trả lời, hãy nói rõ là "Tài liệu hiện tại không chứa đủ thông tin để trả lời câu hỏi này", tuyệt đối không tự bịa ra kiến thức ngoài.

Ngữ cảnh (Context):
{context}

Câu hỏi của người dùng (Query):
{query}

Câu trả lời:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "query"]
)

In [8]:
# 2. Khởi tạo LLM với Groq
# Llama-3-70B hiện là một trong những model mã nguồn mở tốt nhất, 
# suy luận logic cực tốt và hỗ trợ tiếng Việt khá ổn.
llm = ChatGroq(
    temperature=0, 
    model_name="llama-3.3-70b-versatile" 
    # Bạn cũng có thể thử: "mixtral-8x7b-32768" hoặc "gemma2-9b-it"
)

In [9]:
# 3. Kết nối thành một Chain
chain = prompt | llm | StrOutputParser()

# 4. Chạy để lấy kết quả
final_answer = chain.invoke({
    "context": context, 
    "query": query
})

In [10]:
print("=== CÂU TRẢ LỜI TỪ GROQ ===")
print(final_answer)

=== CÂU TRẢ LỜI TỪ GROQ ===
Tài liệu hiện tại không chứa đủ thông tin để trả lời câu hỏi này. Các thông tin cung cấp chủ yếu liên quan đến các loại thuốc và hướng dẫn sử dụng, không có thông tin về việc viết lá thư xin nghỉ học. Nếu bạn cần giúp đỡ về một chủ đề khác liên quan đến thông tin trong ngữ cảnh, vui lòng cho tôi biết.
